# 07 · 投机解码：什么时候它是负收益

第 02 章得出过一个结论：**decode 阶段 GPU 大量时间在等显存，算力是闲置的。**

投机解码（speculative decoding）就是冲着这个浪费去的。核心思路一句话：

> 用一个便宜的模型连续猜 γ 个 token，再用目标模型**一次并行验证**，猜对的直接收下。

但这一章的重点不是"它有多快"，而是**它在什么条件下会变慢**。这是面试官最爱追问的地方，也是很多论文不会告诉你的部分。

In [ ]:
# ===== 引导单元：环境检查 + 测量工具 + MiniGPT（每章自带，直接运行）=====
# 说明：本单元在每个 notebook 里都有一份完整副本，目的是让任何一个 notebook
#       都能在 Colab 里零配置独立运行。想改模型结构，请改 tools/build_notebooks.py
#       里的 SETUP_CODE，然后重跑编译脚本。
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# MiniGPT 只有 2700 万参数，用 float16 跑在 GPU 上；CPU 上 float16 很慢，用 float32
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


def sync():
    """GPU 是异步执行的，计时前必须同步，否则测到的是下发时间不是执行时间。"""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def bench(fn, warmup=3, iters=10):
    """返回单次调用的平均耗时（毫秒）。warmup 用来排除首次 kernel 编译等开销。"""
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters * 1000.0


def peak_mem_mb():
    """当前 CUDA 峰值显存占用（MB）。"""
    if DEVICE != "cuda":
        return 0.0
    return torch.cuda.max_memory_allocated() / 1024 ** 2


def reset_peak():
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()


class Config:
    def __init__(self, vocab_size=50257, block_size=1024, n_layer=4, n_head=6, n_embd=384):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head


class CausalSelfAttention(nn.Module):
    """因果自注意力，支持 KV cache。

    past_kv 传入历史的 (k, v)，本步只为新 token 计算 Q/K/V，然后拼在历史后面。
    返回 (输出, 更新后的 (k, v))，其中 k/v 的 shape 是 (B, n_head, 总长度, head_dim)。
    """

    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.head_dim = cfg.head_dim
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x, past_kv=None, attn_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)

        S = k.size(2)  # 总长度 = 历史 + 本步新增
        if attn_mask is None:
            # 默认因果掩码：本步第 i 个 query 的绝对位置是 S-T+i，只能看见 <= 它的 key
            mask = torch.ones(T, S, device=x.device).tril(diagonal=S - T).bool()
        else:
            # 外部传入的掩码，用于一个 batch 里混合不同进度的序列（第 04、06 章）
            mask = attn_mask
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y), (k, v)


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, past_kv=None, attn_mask=None):
        h, present = self.attn(self.ln_1(x), past_kv, attn_mask)
        x = x + h
        x = x + self.mlp(self.ln_2(x))
        return x, present


class MiniGPT(nn.Module):
    """极简 GPT，结构与 Llama 同源：pre-norm + 因果注意力 + 4 倍扩张 MLP + 权重共享。

    与 Llama 的两处差异：
      - 用可学习位置编码代替 RoPE（简化实现，不影响调度实验的结论）
      - 没有 GQA（本仓库是 MHA，第 03 章会手工比较两者的 KV cache 大小）
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight  # 权重共享，省一份 embedding 参数

        def init(m):
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

        self.apply(init)

    def forward(self, idx, past_kvs=None, pos_offset=0, attn_mask=None):
        """idx: (B, T) 的 token id。

        past_kvs: 长度等于层数的列表，每项是 (k, v)；None 表示从零开始（prefill）。
        pos_offset: 本次输入的第一个 token 的绝对位置。传 int 表示整个 batch 用同一个
                    偏移；传 shape (B,) 的张量表示每条序列各用各的偏移——当 batch 里
                    混合了不同进度的请求时必须这样传。
        attn_mask: 可选的自定义注意力掩码，用于屏蔽填充位。
        """
        B, T = idx.shape
        if torch.is_tensor(pos_offset):
            pos = pos_offset.view(B, 1) + torch.arange(T, device=idx.device)[None, :]
        else:
            pos = torch.arange(pos_offset, pos_offset + T, device=idx.device)[None, :].expand(B, T)
        x = self.wte(idx) + self.wpe(pos)

        presents = []
        for i, blk in enumerate(self.blocks):
            past = None if past_kvs is None else past_kvs[i]
            x, present = blk(x, past, attn_mask)
            presents.append(present)
        return self.lm_head(self.ln_f(x)), presents

    @property
    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def build_model(seed=0, device=DEVICE, dtype=DTYPE, **kw):
    torch.manual_seed(seed)
    cfg = Config(**kw)
    model = MiniGPT(cfg).to(device=device, dtype=dtype)
    return model.eval()


@torch.no_grad()
def generate_naive(model, idx, max_new_tokens):
    """不用 KV cache：每一步都把完整序列重新算一遍（O(n^2) 重算）。"""
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -model.cfg.block_size:])
        idx = torch.cat([idx, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return idx


@torch.no_grad()
def generate_cached(model, idx, max_new_tokens):
    """用 KV cache：prompt 只 prefill 一次，之后每步只喂 1 个 token。"""
    logits, past = model(idx)
    nxt = logits[:, -1].argmax(-1, keepdim=True)
    out = [nxt]
    pos = idx.size(1)
    for _ in range(max_new_tokens - 1):
        logits, past = model(nxt, past_kvs=past, pos_offset=pos)
        pos += 1
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        out.append(nxt)
    return torch.cat([idx] + out, dim=1)


def kv_bytes(n_layer, n_kv_head, head_dim, seq_len, batch=1, dtype_bytes=2):
    """KV cache 字节数。注意是 2（K 和 V 各一份）。"""
    return 2 * n_layer * n_kv_head * head_dim * seq_len * batch * dtype_bytes


print(f"引导单元加载完成 | device={DEVICE} dtype={DTYPE} torch={torch.__version__}")
# ===== 引导单元结束 =====

## 一、机制拆解

一轮投机解码分三步：

1. **草稿（draft）**：便宜地连续生成 γ 个候选 token。
2. **验证（verify）**：把这 γ 个候选**一次性**喂给目标模型。因为是一次前向，GPU 并行处理，所以耗时接近单次 decode——**这是整个技术成立的前提**。
3. **接受（accept）**：从第一个位置开始逐个比对，接受最长的匹配前缀。第一个不匹配的位置用目标模型的输出纠正。

关键收益来源：验证阶段一次前向能确认**多个** token，而正常的 decode 一次前向只能确认一个。

## 二、先验证一个前提：验证真的很便宜吗

"一次前向验证 γ 个 token，耗时接近一次 decode"——这是整个技术的基石。先实测它。

In [ ]:
model = build_model(block_size=4096)
CTX = 256

ctx = torch.randint(0, model.cfg.vocab_size, (1, CTX), device=DEVICE)
_, past = model(ctx)

print(f"上下文长度 {CTX}\n")
print(f"{'一次前向处理的 token 数':>24}{'耗时(ms)':>12}{'相对 1 token':>14}")
print("-" * 52)
base = None
for k in [1, 2, 4, 8, 16, 32]:
    inp = torch.randint(0, model.cfg.vocab_size, (1, k), device=DEVICE)
    ms = bench(lambda: model(inp, past_kvs=past, pos_offset=CTX), warmup=3, iters=10)
    if base is None:
        base = ms
    print(f"{k:>24}{ms:>12.3f}{ms / base:>13.2f}x")

print()
print("这就是全部秘密：token 数从 1 涨到 32，耗时只涨了很小的比例。")
print("因为 decode 阶段真正的瓶颈是把全部权重从显存读一遍——读一次是这些时间，")
print("读一次算 1 个 token 还是算 32 个 token，差别很小（直到算力成为新瓶颈）。")
print()
print("所以：只要验证这一次前向的价格 ≈ 一次普通 decode，而它能确认多个 token，就赚了。")

## 三、实现

下面实现一个贪心版投机解码。为了把算法逻辑跑通并且**验证它输出正确**，这里提供两种草稿：

- `draft="self"`：用目标模型自己当草稿。它一定猜得准（接受率 100%），但**完全不便宜**。
- `draft="random"`：随机猜。它几乎免费，但**完全不准**。

真实的草稿模型要同时满足"便宜"和"准"，这两个极端都做不到——这正是本章要传达的重点。

In [ ]:
@torch.no_grad()
def spec_decode(model, prompt, max_new, gamma=4, draft="self", seed=0):
    """贪心版投机解码。返回 (完整序列, 统计信息)。"""
    g = torch.Generator().manual_seed(seed)
    logits, past = model(prompt)
    pos = prompt.size(1)                              # KV 已覆盖的位置数
    nxt = logits[:, -1].argmax(-1, keepdim=True)      # 位置 pos 上的待确认 token
    out = []                                          # 已确认的 token
    stats = {"proposed": 0, "accepted": 0, "forwards": 1}

    while len(out) + 1 < max_new:
        # ---------- 1) 草稿阶段 ----------
        if draft == "self":
            cands = [nxt]                             # 当前分布的最优点就是第一个猜测
            d_past, d_pos, d_in = past, pos, nxt
            for _ in range(gamma - 1):
                lg, d_past = model(d_in, past_kvs=d_past, pos_offset=d_pos)
                d_pos += 1
                d_in = lg[:, -1].argmax(-1, keepdim=True)
                cands.append(d_in)
            stats["forwards"] += gamma - 1
        else:
            cands = [torch.randint(0, model.cfg.vocab_size, (1, 1), generator=g).to(DEVICE)
                     for _ in range(gamma)]

        cand = torch.cat(cands, dim=1)                # (1, gamma)

        # ---------- 2) 验证阶段：一次前向验证全部候选 ----------
        v_logits, v_past = model(cand, past_kvs=past, pos_offset=pos)
        stats["forwards"] += 1
        # 目标模型在每个候选位置上的贪心输出
        target = torch.cat([nxt, v_logits[:, :-1].argmax(-1, keepdim=True)], dim=1)
        # 全部接受时白送的那个 token
        bonus = v_logits[:, -1].argmax(-1, keepdim=True)

        # ---------- 3) 接受最长匹配前缀 ----------
        a = 0
        for i in range(gamma):
            if target[0, i].item() == cand[0, i].item():
                a += 1
            else:
                break
        stats["proposed"] += gamma
        stats["accepted"] += a

        # ---------- 4) 收下已接受的 token ----------
        for i in range(a):
            out.append(cand[:, i:i + 1])

        # ---------- 5) 纠偏 token：全接受时拿 bonus，否则用目标模型的纠正 ----------
        forced = bonus if a == gamma else target[:, a:a + 1]
        out.append(forced)

        # ---------- 6) KV 截断到接受长度，并补一步让 forced 进入 KV ----------
        keep = pos + a
        past = [(k[:, :, :keep], v[:, :, :keep]) for k, v in v_past]
        lg, past = model(forced, past_kvs=past, pos_offset=keep)
        stats["forwards"] += 1
        pos = keep + 1
        nxt = lg[:, -1].argmax(-1, keepdim=True)

    if len(out) < max_new:
        out.append(nxt)
    return torch.cat([prompt] + out[:max_new], dim=1), stats

## 四、先证明它算得对

贪心验证的投机解码，输出必须和普通贪心解码**逐位完全一致**。不接受任何近似。

In [ ]:
prompt = torch.randint(0, model.cfg.vocab_size, (1, 64), device=DEVICE)
MAX_NEW = 32

ref = generate_cached(model, prompt, MAX_NEW)

print(f"{'草稿类型':<12}{'γ':>4}{'输出与贪心一致':>16}{'接受率':>10}{'前向次数':>10}")
print("-" * 54)
for draft in ["self", "random"]:
    for gamma in [2, 4, 8]:
        got, st = spec_decode(model, prompt, MAX_NEW, gamma=gamma, draft=draft)
        ok = torch.equal(got, ref)
        acc = st["accepted"] / max(1, st["proposed"]) * 100
        print(f"{draft:<12}{gamma:>4}{str(ok):>16}{acc:>9.1f}%{st['forwards']:>10}")

**两种草稿的输出都和贪心完全一致**——算法逻辑是对的。

但注意接受率和前向次数的差别：

- `self`：接受率 100%，但为了猜 γ 个 token 已经花掉了 γ-1 次前向，加上验证和补步，**总前向次数和直接解码差不多**。
- `random`：几乎全部被拒绝，每轮只能推进 1 个 token，却花了验证 + 补步 2 次前向——**比直接解码还慢一倍**。

## 五、实测速度

In [ ]:
ms_base = bench(lambda: generate_cached(model, prompt, MAX_NEW), warmup=1, iters=3)
print(f"基线（普通贪心解码）: {ms_base:8.1f} ms\n")
print(f"{'草稿类型':<12}{'γ':>4}{'耗时(ms)':>12}{'加速比':>10}")
print("-" * 40)
for draft in ["self", "random"]:
    for gamma in [2, 4, 8]:
        ms = bench(lambda: spec_decode(model, prompt, MAX_NEW, gamma=gamma, draft=draft),
                   warmup=1, iters=3)
        print(f"{draft:<12}{gamma:>4}{ms:>12.1f}{ms_base / ms:>9.2f}x")

print()
print("两个极端都拿不到收益，甚至明显变慢。这不是实现问题，是数学约束。")

## 六、收益的理论边界

把上面的直觉写成公式。设：

- `α`：草稿的接受率
- `γ`：每轮草稿长度
- `c_d`：草稿单步成本 ÷ 目标模型单步成本（草稿有多便宜）

一轮投机解码的**期望产出**（标准结论）：

```
E[tokens] = (1 - α^(γ+1)) / (1 - α)
```

一轮的**成本**（以目标模型单步为 1 个单位）：

```
cost = γ × c_d + 1        ← 1 是那次并行验证，它约等于一次普通 decode
```

所以 `加速比 = E[tokens] / cost`。下面把这张表扫出来。

In [ ]:
def expected_tokens(alpha, gamma):
    if alpha >= 1.0:
        return gamma + 1
    return (1 - alpha ** (gamma + 1)) / (1 - alpha)


def speedup(alpha, gamma, c_d):
    return expected_tokens(alpha, gamma) / (gamma * c_d + 1)


print("加速比（c_d = 0.1，草稿模型比目标模型便宜 10 倍）\n")
alphas = [0.2, 0.4, 0.6, 0.8, 0.95]
print(f"{'接受率 α':>10}" + "".join(f"{'γ=' + str(g):>10}" for g in [2, 4, 8]))
print("-" * 40)
for a in alphas:
    row = "".join(f"{speedup(a, g, 0.1):>10.2f}" for g in [2, 4, 8])
    print(f"{a:>10.2f}{row}")

In [ ]:
print("草稿成本 c_d 的影响（α = 0.7, γ = 4）\n")
print(f"{'c_d':>8}{'草稿相对成本':>16}{'加速比':>10}")
print("-" * 36)
for c_d in [0.02, 0.05, 0.1, 0.2, 0.5, 1.0]:
    print(f"{c_d:>8.2f}{c_d * 4:>15.2f}{speedup(0.7, 4, c_d):>10.2f}")

print()
print("读表得到三条结论：")
print("  1. 接受率越低，可行区间越窄：α=0.2 时只有 γ=2 勉强持平（1.03x），γ 越大亏得越多。")
print("  2. γ 不是越大越好：α 低时增大 γ 只会让成本线性上升、产出几乎不涨。")
print("  3. c_d = 1（草稿和目标是同一个模型）时加速比恰好是 1.0——")
print("     这正好对上前面 self 草稿的实测结果。理论和实验对上了。")

## 七、什么时候是负收益

把上面两张表翻译成工程判断：

| 条件 | 为什么是负收益 |
|---|---|
| 草稿模型太弱，接受率低 | 每轮只推进 1 个 token，却付出了验证开销，纯亏 |
| 草稿模型太大，成本接近目标模型 | 成本翻倍，产出不涨（实测里 `self` 就是这个情况） |
| **decoding batch 已经很大** | GPU 算力已经打满，验证那次前向不再"近乎免费"，收益消失 |
| 输出高度随机（高温采样、开放创作） | 接受率天然低 |
| 请求本来就是算力受限（超长 prefill） | 瓶颈不在 decode，优化错了地方 |

第三条尤其值得强调：**投机解码和 continuous batching 抢的是同一份闲置算力。** 并发低时投机解码收益明显；并发高到算力饱和时，投机解码反而降低整体吞吐。这是一个非常容易被忽略的取舍，能主动讲出来会加分很多。

## 八、面试话术

**问：投机解码的收益从哪来？什么时候会变慢？**

- **原理**：decode 是访存受限，算力闲置。草稿模型猜 γ 个 token，目标模型一次前向并行验证，等于用一次前向确认多个 token。
- **成立前提**：验证那一次前向的成本要接近普通 decode。实测显示 token 数从 1 涨到 32，耗时只涨很小比例——因为权重只读了一遍。
- **收益公式**：`E[tokens] = (1-α^(γ+1))/(1-α)`，成本 `γ·c_d + 1`。α 和 c_d 两个参数决定一切。
- **负收益场景**：草稿太弱（α 低）或太贵（c_d 接近 1）；**并发已经打满算力时**，验证不再免费。
- **工程现实**：生产上常用 n-gram / prompt lookup 做草稿（零模型成本，适合有大量重复文本的场景），或者用 MTP/EAGLE 这类把草稿能力直接训进模型的方法。

**作业**

1. 把 `MAX_NEW` 改成 128，重跑实测。投机解码的收益是随生成长度变大还是变小？为什么？
2. 用 `speedup()` 算出：α=0.5、c_d=0.1 时，γ 取多少最优？（提示：扫描 γ=1..16）
3. 思考题：如果服务同时跑着 64 路并发，算力已经打满，此时开启投机解码会发生什么？你会在什么并发区间开启它？

**下一章**：从精度下手——把权重和 KV cache 压到 8 位甚至 4 位。